In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_dir=r'C://Users/kerrie/Documents/02_LocalData/nclimgrid_monthly/'

In [3]:
# info for grid cell selection
city_name = ['San Diego', 'Tucson', 'El Paso', 'Austin', 'Houston', 'New Orleans', 'Mobile', 'Augusta', 'Jacksonville', 'Orlando',]
city_lat = 	[32.810, 32.200, 31.77, 30.27, 29.76, 29.95, 30.68, 33.47, 30.33, 28.500 ]
city_lon = 	[-117.140, -110.890, -106.48, -97.74, -95.36, -90.08, -88.04, -81.97, -81.65, -81.370]
lat_min, lat_max = 28,34
lon_min, lon_max = -117.5,-81

# time vars
year1_start, year2_start, year_end = '1950','1949','2024'
base_start, base_end = '1991','2020'

# Prepare pr file 1 (8 cities)

In [4]:
# get nclimgrid monthly pr
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year1_start,year_end),lat=slice(lat_max,lat_min),lon=slice(lon_min,lon_max))
ds

<xarray.Dataset> Size: 454MB
Dimensions:  (time: 900, lat: 144, lon: 876)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2024-12-01
  * lat      (lat) float32 576B 33.98 33.94 33.9 33.85 ... 28.1 28.06 28.02
  * lon      (lon) float32 4kB -117.5 -117.4 -117.4 ... -81.1 -81.06 -81.02
Data variables:
    prcp     (time, lat, lon) float32 454MB ...
Attributes: (12/14)
    date_created:              2025-01-07 12:54:24.863985
    date_modified:             2025-01-07 12:54:24.864077
    Conventions:               CF-1.6, ACDD-1.3
    ncei_template_version:     NCEI_NetCDF_Grid_Template_v2.0
    title:                     nClimGrid
    naming_authority:          gov.noaa.ncei
    ...                        ...
    geospatial_lat_min:        24.562532
    geospatial_lat_max:        49.3542
    geospatial_lon_min:        -124.6875
    geospatial_lon_max:        -67.020836
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east

In [5]:
# select single grid for each city, save grid lat/lon, calc anomalies

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat[:-2],city_lon[:-2])):
    print('processing',city_name[i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # drop unnecessary coordinates
    city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

print(lat)
print(lon)
print(pr_anom[0])

processing San Diego
processing Tucson
processing El Paso
processing Austin
processing Houston
processing New Orleans
processing Mobile
processing Augusta
[array(32.812534, dtype=float32), array(32.187534, dtype=float32), array(31.770866, dtype=float32), array(30.270866, dtype=float32), array(29.770866, dtype=float32), array(29.937532, dtype=float32), array(30.687532, dtype=float32), array(33.4792, dtype=float32)]
[array(-117.145836, dtype=float32), array(-110.895836, dtype=float32), array(-106.479164, dtype=float32), array(-97.729164, dtype=float32), array(-95.354164, dtype=float32), array(-90.0625, dtype=float32), array(-88.020836, dtype=float32), array(-81.979164, dtype=float32)]
<xarray.DataArray 'prcp' (time: 900)> Size: 4kB
array([ 2.42607422e+01, -2.98775101e+01, -1.54370766e+01, -1.08899746e+01,
        9.44726467e-01, -1.53860676e+00,  7.39941418e-01, -6.22330725e-01,
       -3.06103516e+00, -1.29635744e+01,  1.14253578e+01, -4.11422195e+01,
       -5.51953125e+00, -4.47583694

In [8]:
# create time-indexed pandas dataframe (columns are pr anomalies by city)
for i,city in enumerate(city_name[:-2]):
    if i==0:
        df = pd.DataFrame(pr_anom[i].data,index=pr_anom[i].time,columns=[city_name[i]])
    else:
        df[city]= pr_anom[i].data
df

,San Diego,Tucson,El Paso,Austin,Houston,New Orleans,Mobile,Augusta
1950-01-01,24.260742,-19.914062,-2.001856,-44.206543,2.015236,-83.495636,-78.833099,-70.114975
1950-02-01,-29.877510,8.528027,-4.027083,42.778973,63.515656,-72.553909,-82.873077,-54.144012
1950-03-01,-15.437077,-12.281836,-5.530696,-48.979851,-67.185448,-4.154068,81.698441,-0.780273
1950-04-01,-10.889975,-7.062012,-3.777702,90.943748,38.897820,23.854523,41.778671,-45.050781
1950-05-01,0.944726,-3.393717,-5.942643,-36.089157,-44.301437,-84.612930,-35.067642,54.651985
...,...,...,...,...,...,...,...,...
2024-08-01,0.568099,-15.562370,-26.614616,-29.506771,-87.673279,-75.429108,-108.971680,-50.434669
2024-09-01,-3.070801,-24.553972,-28.582649,-77.203812,-85.412109,279.034424,13.501266,74.068291
2024-10-01,-13.003613,-16.139616,-15.280859,-105.855629,-103.408691,-29.619431,-69.394012,-68.956085
2024-11-01,-18.404720,-2.849024,10.365202,-20.089775,-17.276726,57.530502,24.919952,47.461098


In [9]:
# transpose so city names are the indexes
df_T = df.T
print(df_T.shape)
df_T.head()

(8, 900)


,1950-01-01,1950-02-01,1950-03-01,1950-04-01,1950-05-01,1950-06-01,1950-07-01,1950-08-01,1950-09-01,1950-10-01,...,2024-03-01,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01
San Diego,24.260742,-29.877510,-15.437077,-10.889975,0.944726,-1.538607,0.739941,-0.622331,-3.061035,-12.963574,...,33.333431,-11.200521,-3.815039,-1.198763,-0.830371,0.568099,-3.070801,-13.003613,-18.404720,-44.571907
Tucson,-19.914062,8.528027,-12.281836,-7.062012,-3.393717,18.549023,47.920734,-40.442253,-11.473894,-15.419889,...,15.957422,11.647949,-4.493327,15.638867,9.600422,-15.562370,-24.553972,-16.139616,-2.849024,-29.774349
El Paso,-2.001856,-4.027083,-5.530696,-3.777702,-5.942643,-11.998503,43.163052,-27.284538,-5.483040,0.799219,...,-4.580501,-3.317741,-7.582292,-4.118620,-10.607456,-26.614616,-28.582649,-15.280859,10.365202,-14.309668
Austin,-44.206543,42.778973,-48.979851,90.943748,-36.089157,-25.963997,-17.873341,-44.046810,21.245407,-91.465981,...,-23.460320,14.113670,11.750687,-33.024544,54.006542,-29.506771,-77.203812,-105.855629,-20.089775,-29.711357
Houston,2.015236,63.515656,-67.185448,38.897820,-44.301437,51.634247,0.513542,-107.863708,-97.572266,-117.839355,...,-3.505760,-1.802376,103.618484,-19.365753,249.633667,-87.673279,-85.412109,-103.408691,-17.276726,-20.919952


In [16]:
# add grid lat/lon info
df_T.insert(loc=0, column='LATITUDE', value=lat[:-2])
df_T.insert(loc=1, column='LONGITUDE', value=lon[:-2])
# add other metadata columns
df_T.insert(loc=2, column='UNITS', value='mm')
df_T.insert(loc=3, column='BASE', value='1991-2020')
df_T.insert(loc=4, column='SOURCE', value='https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00332')
df_T.insert(loc=5, column='ACCESSED', value='MARCH 2025')
df_T.head()

,LATITUDE,LONGITUDE,UNITS,BASE,SOURCE,ACCESSED,1950-01-01 00:00:00,1950-02-01 00:00:00,1950-03-01 00:00:00,1950-04-01 00:00:00,...,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00
San Diego,32.812534,-117.145836,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,24.260742,-29.877510,-15.437077,-10.889975,...,33.333431,-11.200521,-3.815039,-1.198763,-0.830371,0.568099,-3.070801,-13.003613,-18.404720,-44.571907
Tucson,32.187534,-110.895836,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,-19.914062,8.528027,-12.281836,-7.062012,...,15.957422,11.647949,-4.493327,15.638867,9.600422,-15.562370,-24.553972,-16.139616,-2.849024,-29.774349
El Paso,31.770866,-106.479164,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,-2.001856,-4.027083,-5.530696,-3.777702,...,-4.580501,-3.317741,-7.582292,-4.118620,-10.607456,-26.614616,-28.582649,-15.280859,10.365202,-14.309668
Austin,30.270866,-97.729164,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,-44.206543,42.778973,-48.979851,90.943748,...,-23.460320,14.113670,11.750687,-33.024544,54.006542,-29.506771,-77.203812,-105.855629,-20.089775,-29.711357
Houston,29.770866,-95.354164,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,2.015236,63.515656,-67.185448,38.897820,...,-3.505760,-1.802376,103.618484,-19.365753,249.633667,-87.673279,-85.412109,-103.408691,-17.276726,-20.919952


In [17]:
# write csv file
df_T.to_csv('pr_anomaly_nclimgrid_monthly_8cities.csv', index_label='CITY')

# Prepare file 2 (2 cities with different start date)

In [18]:
# get nclimgrid monthly pr
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year2_start,year_end),lat=slice(lat_max,lat_min),lon=slice(lon_min,lon_max))
ds

<xarray.Dataset> Size: 460MB
Dimensions:  (time: 912, lat: 144, lon: 876)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1949-01-01 1949-02-01 ... 2024-12-01
  * lat      (lat) float32 576B 33.98 33.94 33.9 33.85 ... 28.1 28.06 28.02
  * lon      (lon) float32 4kB -117.5 -117.4 -117.4 ... -81.1 -81.06 -81.02
Data variables:
    prcp     (time, lat, lon) float32 460MB ...
Attributes: (12/14)
    date_created:              2025-01-07 12:54:24.863985
    date_modified:             2025-01-07 12:54:24.864077
    Conventions:               CF-1.6, ACDD-1.3
    ncei_template_version:     NCEI_NetCDF_Grid_Template_v2.0
    title:                     nClimGrid
    naming_authority:          gov.noaa.ncei
    ...                        ...
    geospatial_lat_min:        24.562532
    geospatial_lat_max:        49.3542
    geospatial_lon_min:        -124.6875
    geospatial_lon_max:        -67.020836
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east

In [23]:
# select single grid for each city, save grid lat/lon, calc anomalies

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat[-2:],city_lon[-2:])):
    print('processing',city_name[-2:][i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # drop unnecessary coordinates
    city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

print(lat)
print(lon)

processing Jacksonville
processing Orlando
[array(30.312532, dtype=float32), array(28.4792, dtype=float32)]
[array(-81.645836, dtype=float32), array(-81.354164, dtype=float32)]


In [24]:
# create time-indexed pandas dataframe (columns are pr anomalies by city)
for i,city in enumerate(city_name[-2:]):
    if i==0:
        df = pd.DataFrame(pr_anom[-2:][i].data,index=pr_anom[-2:][i].time,columns=[city_name[-2:][i]])
    else:
        df[city]= pr_anom[-2:][i].data
df

,Jacksonville,Orlando
1949-01-01,-54.618004,-58.896027
1949-02-01,58.635582,-37.726433
1949-03-01,-62.411949,-59.684898
1949-04-01,51.979652,19.977280
1949-05-01,-33.402931,-56.482132
...,...,...
2024-08-01,111.819824,30.451309
2024-09-01,138.871613,20.322983
2024-10-01,-32.892349,87.212502
2024-11-01,-30.605209,-21.253971


In [25]:
# transpose so city names are the indexes
df_T = df.T
print(df_T.shape)
df_T.head()

(2, 912)


,1949-01-01,1949-02-01,1949-03-01,1949-04-01,1949-05-01,1949-06-01,1949-07-01,1949-08-01,1949-09-01,1949-10-01,...,2024-03-01,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01
Jacksonville,-54.618004,58.635582,-62.411949,51.979652,-33.402931,-72.390228,-35.642746,107.760254,85.411652,-42.241959,...,61.688637,-27.230309,-10.912697,-94.11972,53.336746,111.819824,138.871613,-32.892349,-30.605209,-32.650032
Orlando,-58.896027,-37.726433,-59.684898,19.977280,-56.482132,9.445633,3.310150,62.130997,12.622787,-36.167381,...,-38.394859,-30.413345,-37.702835,-36.02507,-38.789459,30.451309,20.322983,87.212502,-21.253971,-24.752995


In [26]:
# add grid lat/lon info
df_T.insert(loc=0, column='LATITUDE', value=lat[-2:])
df_T.insert(loc=1, column='LONGITUDE', value=lon[-2:])
# add other metadata columns
df_T.insert(loc=2, column='UNITS', value='mm')
df_T.insert(loc=3, column='BASE', value='1991-2020')
df_T.insert(loc=4, column='SOURCE', value='https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00332')
df_T.insert(loc=5, column='ACCESSED', value='MARCH 2025')
df_T.head()

,LATITUDE,LONGITUDE,UNITS,BASE,SOURCE,ACCESSED,1949-01-01 00:00:00,1949-02-01 00:00:00,1949-03-01 00:00:00,1949-04-01 00:00:00,...,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00
Jacksonville,30.312532,-81.645836,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,-54.618004,58.635582,-62.411949,51.979652,...,61.688637,-27.230309,-10.912697,-94.11972,53.336746,111.819824,138.871613,-32.892349,-30.605209,-32.650032
Orlando,28.4792,-81.354164,mm,1991-2020,https://www.ncei.noaa.gov/access/metadata/land...,MARCH 2025,-58.896027,-37.726433,-59.684898,19.977280,...,-38.394859,-30.413345,-37.702835,-36.02507,-38.789459,30.451309,20.322983,87.212502,-21.253971,-24.752995


In [27]:
# write csv file
df_T.to_csv('pr_anomaly_nclimgrid_monthly_2cities.csv', index_label='CITY')

# Test on additional cities



In [33]:
city_name = ['Seattle',	'Portland',	'Boise']
city_lat = [47.608013, 45.5370, 43.618881]
city_lon = [-122.31, -122.6500, -116.215019]

In [34]:
# ds.prcp.isel(time=0).sel(lat=slice(47.7,47.5),lon=slice(-122.4,-122.3)).plot()


In [35]:
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year1_start,year_end),lat=slice(48,43.5),lon=slice(-122.5,-116))

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat,city_lon)):
    print('processing',city_name[i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # drop unnecessary coordinates
    city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

# create time-indexed pandas dataframe (columns are pr anomalies by city)
for i,city in enumerate(city_name):
    if i==0:
        df = pd.DataFrame(pr_anom[i].data,index=pr_anom[i].time,columns=[city_name[i]])
    else:
        df[city]= pr_anom[i].data

# transpose so city names are the indexes
df_T = df.T
print(df_T.shape)
df_T.head()

processing Seattle
processing Portland
processing Boise
(3, 900)


,1950-01-01,1950-02-01,1950-03-01,1950-04-01,1950-05-01,1950-06-01,1950-07-01,1950-08-01,1950-09-01,1950-10-01,...,2024-03-01,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01
Seattle,58.654327,56.867157,82.017418,-13.836037,-28.796909,-26.026302,6.074739,14.565332,1.363476,79.163086,...,-45.963051,-39.845802,-7.536167,-5.165951,-14.284636,20.485254,-24.836720,-18.947266,-33.480499,-1.396347
Portland,134.025253,71.325912,39.602676,-20.653740,-52.662369,4.403580,11.301042,-1.430111,3.381577,136.494049,...,-35.256699,-39.304131,13.488022,-9.405991,-10.768294,3.190006,-1.807877,-9.535255,28.915009,15.433167
Boise,25.427410,1.983528,33.763802,-18.237110,-7.147820,3.594400,-4.267025,5.671191,11.013900,-7.072233,...,15.093880,24.142773,-2.138054,-7.645834,-3.146907,-3.548535,-2.676530,-13.422819,9.259731,28.319889


In [36]:
df


,Seattle,Portland,Boise
1950-01-01,58.654327,134.025253,25.427410
1950-02-01,56.867157,71.325912,1.983528
1950-03-01,82.017418,39.602676,33.763802
1950-04-01,-13.836037,-20.653740,-18.237110
1950-05-01,-28.796909,-52.662369,-7.147820
...,...,...,...
2024-08-01,20.485254,3.190006,-3.548535
2024-09-01,-24.836720,-1.807877,-2.676530
2024-10-01,-18.947266,-9.535255,-13.422819
2024-11-01,-33.480499,28.915009,9.259731


In [37]:
# base period for these anomalies is 1991-2020
enso = pd.read_csv(r'../NOAA_ClimateIndex/Nino34ClimateIndex.txt',
                   header=None,
                   skiprows=1,
                   skipfooter=3,
                   na_values=-99.99,
                   delimiter=r'\s+',
                   names=np.arange(1,13),
                   engine='python')
enso

,1,2,3,4,5,6,7,8,9,10,11,12
1948,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1950,-1.99,-1.69,-1.42,-1.54,-1.75,-1.27,-1.01,-0.97,-0.98,-1.03,-1.23,-1.31
1951,-1.30,-1.04,-0.38,-0.23,-0.01,0.00,0.30,0.17,0.51,0.49,0.55,0.31
1952,0.13,-0.01,-0.11,-0.02,-0.14,-0.54,-0.76,-0.56,-0.36,-0.46,-0.78,-0.39
...,...,...,...,...,...,...,...,...,...,...,...,...
2021,-1.00,-1.00,-0.80,-0.72,-0.46,-0.28,-0.39,-0.53,-0.55,-0.94,-0.94,-1.06
2022,-0.94,-0.89,-0.97,-1.11,-1.11,-0.75,-0.69,-0.97,-1.07,-0.99,-0.90,-0.85
2023,-0.72,-0.46,-0.11,0.14,0.46,0.84,1.02,1.35,1.60,1.72,2.02,2.03
2024,1.81,1.52,1.13,0.77,0.23,0.17,0.04,-0.12,-0.26,-0.27,-0.25,-0.60


In [38]:
enso = enso.dropna(how='any')
enso.shape

(75, 12)

In [39]:
enso_arr = enso.to_numpy().flatten()
type(enso_arr), enso_arr.shape

(numpy.ndarray, (900,))

In [40]:
# enso.describe()

In [41]:
year_start = str(enso.index[0])
year_end = str(enso.index[-1])
year_start,year_end

('1950', '2024')

In [ ]:
# cities to pull out grid cells for
# city_name = ['San Diego','Tucson','San Antonio','Orlando','Nashville','Louisville','Columbus','St. Louis']
# city_lat = 	[32.810, 32.200, 29.460, 28.500, 36.170, 38.220, 39.990, 38.627]
# city_lon = 	[-117.140, -110.890, -98.510, -81.370, -86.780, -85.740, -82.990, -90.199]

city_name = ['San Diego', 'Tucson', 'El Paso', 'Austin', 'Houston', 'New Orleans', 'Mobile', 'Augusta', 'Jacksonville', 'Orlando',]
city_lat = 	[32.810, 32.200, 31.77, 30.27, 29.76, 29.95, 30.68, 33.47, 30.33, 28.500 ]
city_lon = 	[-117.140, -110.890, -106.48, -97.74, -95.36, -90.08, -88.04, -81.97, -81.65, -81.370]

In [ ]:
# ds.prcp.isel(time=0).sel(lat=slice(31.8,31.7),lon=slice(-106.55,-106.4)).plot()

In [ ]:
# get precip
ds=xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year_start,year_end),lat=slice(34,28),lon=slice(-117.5,-81))
ds

# Subset to only 8 cities and calc anomalies

In [ ]:
# city_pr = ds.prcp.sel(lat=city_lat[0],lon=city_lon[0],method='nearest')
# lat = city_pr.lat.data
# lon = city_pr.lon.data
# city_pr = city_pr.drop_vars(['lat','lon'])
# pr_anom = city_pr.groupby('time.month') - city_pr.sel(time=slice('1991','2020')).groupby('time.month').mean('time')
# pr_anom
# # city_pr

In [ ]:
# lat

In [ ]:
lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat,city_lon)):
    print('processing',city_name[i])
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)
    city_pr = city_pr.drop_vars(['lat','lon'])
    # pr_anom.append(city_pr)
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice('1991','2020')).groupby('time.month').mean('time'))

print(lat)
print(lon)
print(pr_anom[0])

In [ ]:
df = pd.DataFrame(pr_anom[0].data,index=pr_anom[0].time,columns=[city_name[0]])
df

In [ ]:
for i,city in enumerate(city_name[1:]):
    df[city]= pr_anom[i+1].data

df

In [ ]:
data = {'LATITUDE':city_lat, 'LONGITUDE':city_lon}
city_info = pd.DataFrame(data,index=city_name)
city_info

In [ ]:
df_T = df.T
print(df_T.shape)
df_T.head()

In [ ]:
df_T.insert(loc=0, column='LATITUDE', value=city_info.LATITUDE)
df_T.head()

In [ ]:
df_T.insert(loc=1, column='LONGITUDE', value=city_info.LONGITUDE)
df_T.head()

In [ ]:
df_T['LATITUDE']=lat
df_T['LONGITUDE']=lon

df_T.head()

In [ ]:
df1 = df_T.loc['San Diego':'Augusta']
df2 = df_T.loc['Jacksonville':'Orlando']

df1.shape,df2.shape

In [ ]:
enso_arr = enso.values.flatten()
enso_arr.shape

In [ ]:
df['enso']=enso_arr
df

In [ ]:
# df['San Diego norm']=(df['San Diego']-df['San Diego'].min())/(df['San Diego'].max()-df['San Diego'].min())
# df['San Diego SD']=(df['San Diego']-df['San Diego'].mean())/df['San Diego'].std()

# df[['San Diego SD','enso']].plot()

In [ ]:
# df[['San Diego','enso']].corr()

In [ ]:
# pr_arr = df['San Diego'].values
# type(pr_arr), pr_arr.shape


In [42]:
nino_inds = np.where(enso_arr>=1)[0]
nina_inds = np.where(enso_arr<=-1)[0]
neut_inds = np.where( (enso_arr>=-.25)&(enso_arr<=.25) )[0]
nino_inds.shape, nina_inds.shape,neut_inds.shape

((82,), (151,), (181,))

In [ ]:
# pr_arr[nino_inds].mean(),pr_arr[nina_inds].mean(),pr_arr[neut_inds].mean()

In [43]:
for city in city_name:
    pr_arr=df[city].values
    print(city,pr_arr[nino_inds].mean(),pr_arr[nina_inds].mean(),pr_arr[neut_inds].mean(),', nino-nina=',round(pr_arr[nino_inds].mean()-pr_arr[nina_inds].mean(),2))    

Seattle -0.8156569 6.235392 2.3386655 , nino-nina= -7.05
Portland 2.2868273 16.921572 6.4322004 , nino-nina= -14.63
Boise 0.6381385 3.2482467 0.50765926 , nino-nina= -2.61


Calculate the difference between mean nino and mean nina precip anomalies for each station
Plot the differences with the xaxis labeled with the city names

Sort stations by nino-nina pr difference value highest to lowest

Plot the sorted differences on a bar plot with xaxis city names


In [ ]:
pr_arr = df.loc[:,'San Diego':'Orlando'].values
pr_arr.shape

In [ ]:
nino_mean = pr_arr[nino_inds,:].mean(axis=0)
nina_mean = pr_arr[nina_inds,:].mean(axis=0)
neut_mean = pr_arr[neut_inds,:].mean(axis=0)
ninoa_diff = nino_mean-nina_mean
ninoa_diff

In [ ]:
# which city has the largest mean precip anomaly during el nino months?
print(city_name[nino_mean.argmax()])

# which city has the largest mean precip anomaly during la nina months?
print(city_name[nina_mean.argmax()])

# which city has the largest difference between mean precip anomalies in nino vs nina months?
print(city_name[ninoa_diff.argmax()])


In [ ]:
fig = plt.figure(figsize=(8,3))
plt.plot(city_name,ninoa_diff,marker='o',lw=0)
plt.xticks(rotation=300,ha='left')
plt.ylabel('Precipitation (mm)')
plt.title('nino minus nina mean monthly precip difference')
plt.show()

In [ ]:
sort_order = np.argsort(ninoa_diff)
sorted_diff = ninoa_diff[sort_order]
sorted_labels = np.array(city_name)[sort_order]

print(sorted_diff)
print(sorted_labels)

In [ ]:
fig = plt.figure(figsize=(8,3))
plt.bar(sorted_labels,sorted_diff)
plt.xticks(rotation=300,ha='left')
plt.ylabel('Precipitation (mm)')
plt.title('nino minus nina mean monthly precip difference')
plt.show()

In [ ]:
# ds.prcp.sel(lat=lat[1],lon=lon[1]).drop_vars(['lat','lon']).to_dataframe()


In [ ]:
# step 9 spatial subset
# clip data to a bounding box

# get clip object
clipobj=gpd.read_file(shpfile)
clipobj.crs

In [ ]:
# assign crs to netcdf data
ds.rio.write_crs("epsg:4326",inplace=True)
ds_clip=ds.rio.clip(clipobj.geometry.apply(shapely.geometry.mapping),clipobj.crs,drop=True,invert=False)

In [ ]:
ds_clip['tmax'] = ds_clip.tmax.round(decimals=2)
ds_clip

In [ ]:
del ds_clip['spatial_ref']
ds_clip

In [ ]:
ds_clip = ds_clip.reindex(lat=ds_clip.lat[::-1])
ds_clip

In [ ]:
tmax=ds_clip.tmax.data.astype('float16')
tmax.shape, tmax.nbytes

In [ ]:
lat= list(ds_clip.lat.data)
lon= list(ds_clip.lon.data)
time=ds_clip.time.data
# lat

In [ ]:
time = time.astype('str')
time = [t[0:7] for t in time]
time[0]

In [ ]:
with open('nclimgrid_tmax_196401-202312.npy','wb') as f:
    np.save(f,tmax)

with open('nclimgrid_tmax_meta_lat.txt','w') as f:
    np.savetxt(f,lat,fmt='%.6f')

with open('nclimgrid_tmax_meta_lon.txt','w') as f:
    np.savetxt(f,lon,fmt='%.6f')

with open('nclimgrid_tmax_meta_time.txt','w') as f:
    np.savetxt(f,time,fmt='%s')

In [ ]:
test=np.load('nclimgrid_tmax.npy')
test.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.contourf(test[0,:,:])

In [ ]:
# ds_clip.tmax.to_netcdf(data_dir+'nclimgrid_tmax_196401-202312.nc')
ds_clip.tmax.to_netcdf(data_dir+'nclimgrid_tmax_199401-202312.nc')

In [ ]:
import numpy as np
              # C:\Users\kerrie\Documents\01_LocalCode\repos\DEV_canvas_beginner_py
datafile = r'C://Users/kerrie/Documents/01_LocalCode/repos/DEV_canvas_beginner_py/tmax_data.csv'

In [ ]:
# data = np.loadtxt(datafile, delimiter=',', skiprows=1)
data = np.loadtxt('tmax_data.csv', delimiter=',', skiprows=1, usecols=(3), unpack=True)


data

In [ ]:
# 1) drop data we don't need"
ds = ds.drop_vars('tavg')

steps 2-7 aren't necessary, data already looks good wrt to these items

In [ ]:
# step 8 millimeter --> mm/day
ds.prcp.attrs['units']='mm/day'

In [ ]:
# step 9 spatial subset
# clip data to a bounding box

# get clip object
box=gpd.read_file(shpfile)

# assign crs to netcdf data
ds.rio.write_crs("epsg:4326",inplace=True)
ds_clip=ds.rio.clip(box.geometry.apply(shapely.geometry.mapping),box.crs,drop=True,invert=False)
ds_clip

In [ ]:
# save metadata separately for later
coords = ds_clip.coords
prcp_attrs = ds_clip.prcp.attrs
tmax_attrs = ds_clip.tmax.attrs
tmin_attrs = ds_clip.tmin.attrs
dims = ds_clip.dims
print(dims)
coords

step 10 & 11, round and reduce precision

In [ ]:
# choosing a single time to test data precision, looking for values on order of at least 100
testtime='2000-06-4'
ds_clip.prcp.sel(time=testtime).plot()

In [ ]:
# float16 is probably not enough precision, let's check on a subset of the data

print('loading float16')
prcp_16 = ds_clip.prcp.sel(time=testtime).astype('float16').load()
print('loading float32')
prcp_32 = ds_clip.prcp.sel(time=testtime).astype('float32').load()
print('loading float64')
prcp_64 = ds_clip.prcp.sel(time=testtime).load()

prcp_16.max().item(),prcp_32.max().item(),prcp_64.max().item()

so we can change the data type from float64 to float32 but not go any smaller

In [ ]:
# steps 10 & 11
ds = ds_clip.round(decimals=2)
ds = ds_clip.astype('float32')
ds

# write files

I think what we have to do to write each chunk to a separate file is:
- chunk xr arrays in space instead of time
- convert to numpy so we can use to_delayed and ravel
- every worker needs the xr metadata for variable and coordinates
- write a dask delayed function that takes the numpy array data and the metadata separately
- inside the dask delayed function re-create the xarray object and write to file

In [ ]:
# create an integer index for the dim we will chunk (longitude)
ilon_ind = np.arange(0,len(ds.lon)).astype('int')
ds.coords['ilon_index']=('lon',ilon_ind)
# ds.ilon_index.attrs['standard_name']='integer_index_longitude'

In [ ]:
# chunking along longitude only
nlons=6
ds = ds.chunk({'time':-1,'lat':-1,'lon':nlons})
ds

In [ ]:
ilon_chunks = xr.DataArray(ds.ilon_index.data,coords={'ilon_index':('lon',ds.ilon_index.data)})#.chunk({'lon':nlons}).data.to_delayed().ravel()
ilon_chunks

In [ ]:
# a function to return a list of data chunks 
# and the corresponding longitude coord chunks
def xr_ds_to_delayed(ds,varname):
    # chunk the appropriate variable in ds, delay, ravel to list
    var_chunks = ds[varname].data.to_delayed().ravel()
    # xarray doesn't allow chunking of coordinates, so we have to make a new variable to chunk
    # the convoluted process is xarray-->numpy-->xarray-->chunk-->numpy-->delay-->ravel
    lon_chunks = xr.DataArray(ds.lon.data,coords={'lon':('lon',ds.lon.data)}).chunk({'lon':nlons}).data.to_delayed().ravel()
    ilon_chunks = xr.DataArray(ds.ilon_index.data,coords={'ilon_index':('lon',ds.ilon_index.data)}).chunk({'lon':nlons}).data.to_delayed().ravel()
    return var_chunks,lon_chunks,ilon_chunks,ds.spatial_ref

# numpy back to xarray, reattaching metadata and writing chunks to separate files
def write_chunk_to_netcdf(datapath,chunk_id,varname,
                          np_datachunk,xr_time,xr_lat,
                          spatial_ref,np_ilonind,np_lonchunk,
                          lon_meta,var_meta):
    
    chunk_id = str(chunk_id).zfill(3)
    # numpy-->xarray
    xr_datachunk = xr.Dataset({varname:(['time','lat','lon'],np_datachunk)},
                              coords={'time':('time',xr_time.data),
                                      'lat':('lat',xr_lat.data),
                                      'lon':('lon',np_lonchunk),
                                      'spatial_ref':('spatial_ref',spatial_ref),
                                      'ilon_index':('lon',np_ilonind)})
    # copy over xr metadata
    xr_datachunk.time.attrs=xr_time.attrs
    xr_datachunk.lat.attrs=xr_lat.attrs
    xr_datachunk.lon.attrs=lon_meta
    xr_datachunk.ilon_index.attrs['standard_name']='integer_index_longitude'
    xr_datachunk[varname].attrs=var_meta
    
    # clean up metadata
    attrslist=['time','lat','lon',varname]
    for att in attrslist:
        if 'valid_min' in xr_datachunk[att].attrs:
            del xr_datachunk[att].attrs['valid_min']
        if 'valid_max' in xr_datachunk[att].attrs:
            del xr_datachunk[att].attrs['valid_max']
        if (att!=varname) and ('comment' in xr_datachunk[att].attrs):
            del xr_datachunk[att].attrs['comment']
        if 'id' in xr_datachunk[att].attrs:
            del xr_datachunk[att].attrs['id']            
    
    # write file
    xr_datachunk.to_netcdf(datapath+'chunkedlon/'+varname+'_nClimGridDaily_USsouth_'+chunk_id+'.nc')
    return chunk_id

In [ ]:
%%time 
var='prcp'
var_chunks,lon_chunks,ilon_chunks,spatial_ref = xr_ds_to_delayed(ds,var)
task_list= [dask.delayed(write_chunk_to_netcdf)(data_dir,id,var,
                                                datachunk,ds.time,ds.lat,
                                                spatial_ref,ilonchunk,lonchunk,
                                                ds.lon.attrs,ds[var].attrs) \
            for id,(datachunk,ilonchunk,lonchunk) in enumerate(zip(var_chunks,ilon_chunks,lon_chunks))]
completed_files = dask.compute(*task_list)
len(completed_files)

In [ ]:
%%time 
var='tmax'
var_chunks,lon_chunks = xr_ds_to_delayed(ds,var)
task_list= [dask.delayed(write_chunk_to_netcdf)(data_dir,id,var,
                                                datachunk,ds.time,ds.lat,
                                                lonchunk,ds.lon.attrs,ds[var].attrs) \
            for id,(datachunk,lonchunk) in enumerate(zip(var_chunks,lon_chunks))]
completed_files = dask.compute(*task_list)
len(completed_files)

In [ ]:
%%time 
var='tmin'
var_chunks,lon_chunks = xr_ds_to_delayed(ds,var)
task_list= [dask.delayed(write_chunk_to_netcdf)(data_dir,id,var,
                                                datachunk,ds.time,ds.lat,
                                                lonchunk,ds.lon.attrs,ds.prcp.attrs) \
            for id,(datachunk,lonchunk) in enumerate(zip(var_chunks,lon_chunks))]
completed_files = dask.compute(*task_list)

In [ ]:
var='prcp'
files = glob.glob(data_dir+var+'_nClimGridDaily_USsouth_*.nc')
test=xr.open_mfdataset(files)
test

In [ ]:
test[var].isel(time=15).plot()

# old code below to write 1 single file per variable

In [ ]:
%%time
filename='prcp_nClimGridDaily_1951-2024_USsouth.nc'
print('writing',filename)
ds.prcp.to_netcdf(data_dir+filename)

In [ ]:
%%time
filename='tmax_nClimGridDaily_1951-2024_USsouth.nc'
print('writing',filename)
ds.tmax.to_netcdf(data_dir+filename)

In [ ]:
%%time
filename='tmin_nClimGridDaily_1951-2024_USsouth.nc'
print('writing',filename)
ds.tmin.to_netcdf(data_dir+filename)

In [ ]:
test=xr.open_mfdataset(data_dir+filename)
test

In [ ]:
client.shutdown()